# Codex Ultra Cloud Launcher (Google Drive Edition)

This notebook mirrors `launch.sh` and exposes the local service through a public tunnel.

- Default service port: `43110`
- Before startup, installs missing CLI tools with `cxu install --all --yes` and displays `cxu doctor` results
- Default tunnel: Cloudflare Quick Tunnel (no account or token required)
- Optional alternatives: `localtunnel` or `ngrok`
- ngrok can use an existing login/configuration; set `NGROK_AUTHTOKEN` only when authentication is needed

> The public URL is reachable by anyone who has it. Do not expose private projects or credentials.
- This Drive edition mounts Google Drive at `/content/drive` first; persistent files can be kept under `/content/drive/MyDrive/codex-ultra`.
- If you do not need Google Drive, use the standard `launch.ipynb` instead.

In [ ]:
# Mount Google Drive for persistent storage (Drive edition only).
# Skip this cell if you use the standard launch.ipynb.
from pathlib import Path

DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_WORKSPACE = DRIVE_MOUNT_POINT / 'MyDrive' / 'codex-ultra'

try:
    from google.colab import drive  # type: ignore
except ImportError:
    print('Not running in Google Colab; skipping Google Drive mount.')
else:
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    DRIVE_WORKSPACE.mkdir(parents=True, exist_ok=True)
    print(f'Google Drive mounted at {DRIVE_MOUNT_POINT}')
    print(f'Persistent workspace: {DRIVE_WORKSPACE}')


In [ ]:
import atexit
import os
import platform
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
from pathlib import Path

PORT = int(os.environ.get('PORT', '43110'))
TUNNEL_PROVIDER = os.environ.get('TUNNEL_PROVIDER', 'cloudflare').lower()
REPO_DIR = Path.cwd()
BIN_DIR = Path.home() / '.local' / 'bin'
BIN_DIR.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'CODEX_UI_TRUST_PROXY': '1',
    'CODEX_UI_ALLOWED_ORIGINS': '*',
    'CODEX_UI_HOST': '0.0.0.0',
    'PORT': str(PORT),
    'CODEX_UI_LEGAL_DIR': os.environ.get('CODEX_UI_LEGAL_DIR', str(REPO_DIR)),
    'PATH': f"/usr/local/bin:{Path.home() / '.local' / 'bin'}:{Path.home() / '.codex-ultra' / 'bin'}:{os.environ.get('PATH', '')}",
})

print(f'Working directory: {REPO_DIR}')
print(f'Codex Ultra port: {PORT}')
print(f'Tunnel provider: {TUNNEL_PROVIDER}')
print('Proxy trust enabled; allowed origins: *')

In [ ]:
def run(command, check=True):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, check=check, text=True)

# Match launch.sh: install cxu only when it is not already available.
if shutil.which('cxu') is None:
    run(['bash', '-lc', 'curl -fsSL https://install.codex-ultra.top/install.sh | sh'])

if shutil.which('codex') is None:
    print('Codex CLI not found; installing the standalone runtime...')
    run(['bash', '-lc', 'curl -fsSL https://chatgpt.com/codex/install.sh | sh'], check=False)

# Background notebook services cannot answer cxu's interactive install prompt.
# Explicitly install missing tools before starting the service.
run(['cxu', 'install', '--all', '--yes'])

# cxu may return success when optional installs fail; show the actual results in the cell.
doctor_result = subprocess.run(['cxu', 'doctor', '--no-prompt'], capture_output=True, text=True)
print(doctor_result.stdout, end='')
if doctor_result.stderr:
    print(doctor_result.stderr, end='', file=sys.stderr)
doctor_result.check_returncode()

print('cxu:', shutil.which('cxu') or 'not found')
print('codex:', shutil.which('codex') or 'not found')

In [ ]:
service_process = None
tunnel_process = None
public_url = None

def port_is_open(host='127.0.0.1', port=PORT):
    with socket.socket() as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

if shutil.which('cxu'):
    service_command = ['cxu', 'serve', '--port', str(PORT)]
elif Path('/opt/codex-ultra/runtime/bun/bin/bun').is_file() and Path('/opt/codex-ultra/server-bundle/index.js').is_file():
    service_command = ['/opt/codex-ultra/runtime/bun/bin/bun', 'run', '/opt/codex-ultra/server-bundle/index.js', '--port', str(PORT)]
else:
    raise RuntimeError('cxu was not installed and no bundled runtime was found')

service_log = open('/tmp/codex-ultra-service.log', 'a', encoding='utf-8')
service_process = subprocess.Popen(service_command, cwd=REPO_DIR, env=os.environ.copy(), stdout=service_log, stderr=subprocess.STDOUT, start_new_session=True)

for _ in range(30):
    if service_process.poll() is not None:
        raise RuntimeError(f'Codex Ultra exited early; see {service_log.name}')
    if port_is_open():
        break
    time.sleep(1)
else:
    raise TimeoutError(f'Codex Ultra did not open port {PORT}; see {service_log.name}')

print(f'Codex Ultra is running on http://127.0.0.1:{PORT}')
print(f'Service log: {service_log.name}')

In [ ]:
def ensure_cloudflared():
    existing = shutil.which('cloudflared')
    if existing:
        return existing
    machine = platform.machine().lower()
    arch = {'x86_64': 'amd64', 'amd64': 'amd64', 'aarch64': 'arm64', 'arm64': 'arm64'}.get(machine)
    if arch is None:
        raise RuntimeError(f'Unsupported CPU architecture for cloudflared: {machine}')
    target = BIN_DIR / 'cloudflared'
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}'
    run(['curl', '-fL', '--retry', '3', '-o', str(target), url])
    target.chmod(0o755)
    return str(target)


def ensure_ngrok():
    existing = shutil.which('ngrok')
    if existing:
        return existing
    machine = platform.machine().lower()
    asset = {'x86_64': 'amd64', 'amd64': 'amd64', 'aarch64': 'arm64', 'arm64': 'arm64'}.get(machine)
    if asset is None:
        raise RuntimeError(f'Unsupported CPU architecture for ngrok: {machine}')
    archive = Path('/tmp/ngrok.zip')
    target = BIN_DIR / 'ngrok'
    url = f'https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-{asset}.zip'
    run(['curl', '-fL', '--retry', '3', '-o', str(archive), url])
    run(['unzip', '-o', str(archive), 'ngrok', '-d', str(BIN_DIR)])
    target.chmod(0o755)
    return str(target)


def start_tunnel():
    global tunnel_process, public_url
    if TUNNEL_PROVIDER in ('none', 'off', 'disabled'):
        print(f'Tunnel disabled. Local URL: http://127.0.0.1:{PORT}')
        return

    if TUNNEL_PROVIDER == 'cloudflare':
        tunnel_command = [ensure_cloudflared(), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate']
        url_pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    elif TUNNEL_PROVIDER == 'localtunnel':
        if shutil.which('npx') is None:
            raise RuntimeError('localtunnel requires Node.js and npx')
        tunnel_command = ['npx', '--yes', 'localtunnel', '--port', str(PORT)]
        url_pattern = re.compile(r'https://[a-z0-9-]+\.loca\.lt')
    elif TUNNEL_PROVIDER == 'ngrok':
        ngrok = ensure_ngrok()
        token = os.environ.get('NGROK_AUTHTOKEN')
        if token:
            subprocess.run([ngrok, 'config', 'add-authtoken', token], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        else:
            print('NGROK_AUTHTOKEN is not set; trying the existing ngrok login/configuration.')
        tunnel_command = [ngrok, 'http', str(PORT), '--log', 'stdout']
        url_pattern = re.compile(r'https://[^\s]+\.ngrok(?:-free)?\.(?:app|com)')
    else:
        raise ValueError('TUNNEL_PROVIDER must be cloudflare, localtunnel, ngrok, or none')

    tunnel_process = subprocess.Popen(tunnel_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    deadline = time.time() + 45
    while time.time() < deadline:
        line = tunnel_process.stdout.readline()
        if line:
            match = url_pattern.search(line)
            if match:
                public_url = match.group(0)
                print(f'Public URL: {public_url}')
                print('Keep this notebook kernel running while you use the URL.')
                return
        elif tunnel_process.poll() is not None:
            break
        time.sleep(0.2)
    raise TimeoutError('The tunnel did not provide a public URL. Check the tunnel process output.')


start_tunnel()

In [ ]:
def cleanup(*_args):
    for process, name in ((tunnel_process, 'tunnel'), (service_process, 'service')):
        if process is not None and process.poll() is None:
            print(f'Stopping {name}...')
            process.terminate()
    if 'service_log' in globals() and not service_log.closed:
        service_log.close()

atexit.register(cleanup)
signal.signal(signal.SIGINT, cleanup)
signal.signal(signal.SIGTERM, cleanup)

print('Launcher is ready.')
print('To stop manually, run: cleanup()')